# Phase 1: Data Engineering & Preprocessing for E-Commerce Purchase Prediction
**Course:** 79747_CO5119_002416_IMP  
**Author:** Justus Izuchukwu Onuh (oijustus.sdh241) | Group 22 AI Core  

## Objective
This notebook executes the data engineering pipeline for a Decision Tree Classifier designed to predict e-commerce purchasing behavior. The raw dataset from Kaggle contains millions of multi-category interactions. To optimize for memory efficiency and domain specificity, this pipeline implements **chunked memory processing** to isolate 'computer' categories, engineer user-session features, and establish a binary classification target.

In [4]:
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Configure visual styling for academic reporting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 150, 'font.size': 12})

print("Environment initialized. Libraries successfully imported.")

Environment initialized. Libraries successfully imported.


## 1. Automated Data Acquisition
To ensure total reproducibility, the notebook connects directly to the Kaggle API to retrieve the `ecommerce-behavior-data-from-multi-category-store` dataset. 

*Note: Execution of this cell requires a valid `kaggle.json` token configured in the host machine's `~/.kaggle/` directory. If the pre-processed data already exists in the local directory, this step will be bypassed to save bandwidth.*

In [7]:
import os
import zipfile
from pathlib import Path

# 1. Programmatically configure the new Kaggle API token
token_value = "KGAT_e205c794d4e53ae96ce6349bcdfa6068"
os.environ['KAGGLE_API_TOKEN'] = token_value

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)

token_file = kaggle_dir / 'access_token'
with open(token_file, 'w') as f:
    f.write(token_value)

# macOS requires this file to have restricted permissions (600)
token_file.chmod(0o600)

# 2. Define dataset parameters
dataset_name = "mkechinov/ecommerce-behavior-data-from-multi-category-store"
raw_data_dir = "../data/raw/"
file_name = "2019-Oct.csv" 
file_path = os.path.join(raw_data_dir, file_name)

os.makedirs(raw_data_dir, exist_ok=True)

# 3. Execute download via Python API
if not os.path.exists(file_path):
    print("Initiating dataset download via Kaggle Python API...")
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate() 
        
        print("Authenticated successfully. Downloading and extracting...")
        print("Note: This is a 5GB file. Depending on your ISP bandwidth (e.g., VNPT, Viettel, or FPT), this process may take several minutes.")
        
        api.dataset_download_files(dataset_name, path=raw_data_dir, unzip=True)
        print("Download and extraction complete.")
        
    except Exception as e:
        print(f"Error during download: {e}")
else:
    print("Raw dataset found locally. Bypassing download.")

Initiating dataset download via Kaggle Python API...
Error during download: No module named 'kaggle'


## 2. Memory-Efficient Data Loading & Domain Filtering
The raw CSV file exceeds 5GB, making standard in-memory loading unstable on standard hardware. We implement an iterative chunking mechanism to read the file in batches of 500,000 rows. During this read process, we strictly filter the data to isolate interactions where the `category_code` contains the keyword **'computers'**.

In [3]:
chunk_size = 500000
processed_chunks = []
processed_data_dir = "../data/processed/"
processed_file = os.path.join(processed_data_dir, "filtered_computers_data.csv")
os.makedirs(processed_data_dir, exist_ok=True)

if not os.path.exists(processed_file):
    print("Processing massive dataset in chunks. This may take a few minutes...")
    
    # Iterate through the massive file
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        # Drop rows missing category_code to avoid string matching errors
        chunk = chunk.dropna(subset=['category_code'])
        
        # Filter strictly for computer-related products
        computer_data = chunk[chunk['category_code'].str.contains('computers', case=False, na=False)]
        processed_chunks.append(computer_data)
        
    # Concatenate all filtered chunks into a single DataFrame
    df_computers = pd.concat(processed_chunks, ignore_index=True)
    
    # Save checkpoint to disk to avoid rerunning chunking in future sessions
    df_computers.to_csv(processed_file, index=False)
    print(f"Filtering complete. Final dataset contains {len(df_computers)} interactions.")
else:
    print("Loading pre-filtered computer dataset...")
    df_computers = pd.read_csv(processed_file)

# Display sample of the isolated domain data
display(df_computers.head())

Processing massive dataset in chunks. This may take a few minutes...


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/2019-Oct.csv'

## 3. Feature Engineering: Session Aggregation
The Decision Tree Classifier requires a tabular format where each row represents a unique `user_session`. We must mathematically derive our predictive features from the raw transactional logs.

**Engineered Features:**
1. **Session Duration:** Time delta between the first and last event in a session (seconds).
2. **Product Views:** Total count of `view` events per session.
3. **Cart Additions:** Total count of `cart` events per session.
4. **Brand Affinity:** The statistical mode (most frequent) brand interacted with during the session.
5. **Price Category:** The maximum price point viewed or added to cart during the session.
6. **Purchased (Target Variable):** A binary flag (1 = purchased, 0 = abandoned) based on the presence of a `purchase` event in the session.

In [ ]:
print("Converting timestamps and engineering session features...")

# Convert event_time to datetime object and strip timezone for calculation
df_computers['event_time'] = pd.to_datetime(df_computers['event_time']).dt.tz_localize(None)

# Define aggregation logic per session
def calculate_duration(x):
    return (x.max() - x.min()).total_seconds()

def count_event(x, event_name):
    return (x == event_name).sum()

def get_brand_mode(x):
    return x.mode()[0] if not x.mode().empty else 'unknown'

# Group by user_session and apply aggregations
df_features = df_computers.groupby('user_session').agg(
    Session_Duration=('event_time', calculate_duration),
    Product_Views=('event_type', lambda x: count_event(x, 'view')),
    Cart_Additions=('event_type', lambda x: count_event(x, 'cart')),
    Target_Purchased=('event_type', lambda x: count_event(x, 'purchase')), 
    Brand_Affinity=('brand', get_brand_mode),
    Max_Price=('price', 'max')
).reset_index()

# Convert Target to strict Binary (1 if >0 purchases, else 0)
df_features['Target_Purchased'] = (df_features['Target_Purchased'] > 0).astype(int)

# Filter out "bounce" sessions (duration = 0 and views = 1) to reduce noise
df_features = df_features[(df_features['Session_Duration'] > 0) | (df_features['Cart_Additions'] > 0)]

print(f"Feature engineering complete. Unique actionable sessions generated: {len(df_features)}")
display(df_features.head())

## 4. Data Cleaning & Encoding
To prepare the engineered dataset for the Scikit-Learn Decision Tree algorithm, we must resolve missing values and transform categorical text data into numerical representations. We will also apply standard scaling to numerical features to optimize model performance and interpretability.

In [ ]:
# 1. Handle Missing Values
# Fill missing brands with 'unknown'
df_features['Brand_Affinity'] = df_features['Brand_Affinity'].fillna('unknown')
# Fill missing prices with the median price to preserve distribution
df_features['Max_Price'] = df_features['Max_Price'].fillna(df_features['Max_Price'].median())

# 2. Categorical Encoding
# Using LabelEncoder for Brand Affinity
le = LabelEncoder()
df_features['Brand_Encoded'] = le.fit_transform(df_features['Brand_Affinity'])

# 3. Drop non-predictive string identifiers
df_final = df_features.drop(columns=['user_session', 'Brand_Affinity'])

# Save the final engineered dataset for Phase 2 (Model Training)
final_dataset_path = os.path.join(processed_data_dir, "model_ready_data.csv")
df_final.to_csv(final_dataset_path, index=False)

print("Data Cleaning and Encoding complete. Dataset is ready for AI Pipeline.")
display(df_final.describe())

## 5. Exploratory Data Visualization
Before training the model, we must validate our engineered features. Below, we visualize the correlation matrix to identify multicollinearity and inspect the distribution of our target variable to assess class imbalance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Target Variable Class Distribution
sns.countplot(data=df_final, x='Target_Purchased', palette='viridis', ax=axes[0])
axes[0].set_title('Distribution of Target Variable (Purchase vs Abandonment)', fontweight='bold')
axes[0].set_xlabel('Purchased (0 = No, 1 = Yes)')
axes[0].set_ylabel('Session Count')

# Add percentage labels to the bars
total = len(df_final)
for p in axes[0].patches:
    percentage = f'{100 * p.get_height() / total:.1f অসুস্থ}%'
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_y() + p.get_height() + (total * 0.01)
    axes[0].annotate(percentage, (x, y), size=12, fontweight='bold')

# Subplot 2: Feature Correlation Heatmap
corr_matrix = df_final.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, 
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

# Additional Visualization: Cart Additions vs Purchase
plt.figure(figsize=(8, 5))
sns.barplot(data=df_final, x='Target_Purchased', y='Cart_Additions', palette='mako')
plt.title('Average Cart Additions by Conversion Outcome', fontweight='bold')
plt.xlabel('Purchased (0 = No, 1 = Yes)')
plt.ylabel('Average Cart Additions per Session')
plt.show()